In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36'}

In [3]:
company_data = []

In [4]:
for i in range(1,10):
    web_page = requests.get(f"https://www.ambitionbox.com/list-of-companies?campaign=homepage_explore_companies_widget&page={i}",headers=headers).text
    soup = BeautifulSoup(web_page, 'html.parser')
    company_list = soup.find_all('div', class_='companyCardWrapper')
    for company in company_list:
        detail = {}
        detail['name'] = company.find('h2', class_='companyCardWrapper__companyName').text.strip()
        detail['rating'] = company.find('div', class_='rating_text rating_text--md').text.strip()
        detail['tags'] = company.find('span', class_='companyCardWrapper__interLinking').text.strip()
        try:
            rated_for = company.find('div', class_='companyCardWrapper__ratingComparisonWrapper')
            all_divs = rated_for.find_all('div')
            for rate_div in all_divs:
                if rate_div.find('span', class_='companyCardWrapper__ratingHeader--high'):
                    rated_for_high = rate_div.find('span', class_='companyCardWrapper__ratingValues')
                    if rated_for_high:
                        detail['high_rated_for'] = rated_for_high.text.strip()
                if rate_div.find('span', class_='companyCardWrapper__ratingHeader--critical'):
                    critical_values = rate_div.find('span', class_='companyCardWrapper__ratingValues')
                    if critical_values:
                        detail['critical_rated_for'] = critical_values.text.strip()
        except Exception as e:
            detail['high_rated_for'] = 'NA'
            detail['critical_rated_for'] = 'NA'
        company_data.append(detail)

In [5]:
df = pd.DataFrame(company_data)
df.to_csv('webscrap_data.csv', index=False) 